# 31_clip_zeroshot.ipynb

**14주차 · 3교시** 실습 노트북

- 이론 설명과 관찰 포인트는 배포 자료(`14week/student/`)를 함께 보세요.
- 실행 환경: `%DL2026_HOME%\venv` 활성화 후 `Python (dl2026)` 커널.
- 전체 8셀. 위에서부터 순서대로 실행합니다.

## 1. 실습 6 — 이미지–텍스트 유사도

**셀 1** — CLIP 로딩

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False

DEV = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(MODEL).to(DEV).eval()
proc  = CLIPProcessor.from_pretrained(MODEL)
print("임베딩 차원 :", model.config.projection_dim)      # 512  ★ 이미지·텍스트 공통

**셀 2** — 이미지 하나, 문장 여러 개 ★

In [ ]:
img = Image.open("outputs/generated/first.png").convert("RGB")   # 2교시 생성 이미지
texts = ["a photo of a cabin in the snow",
         "a photo of a beach in summer",
         "a photo of a city street at night",
         "a photo of a dog"]

inputs = proc(text=texts, images=img, return_tensors="pt", padding=True).to(DEV)
with torch.no_grad():
    out = model(**inputs)

print("이미지 임베딩 :", out.image_embeds.shape)     # (1, 512)
print("텍스트 임베딩 :", out.text_embeds.shape)      # (4, 512)  ★ 같은 512

probs = out.logits_per_image.softmax(dim=-1)[0]
for t, p in zip(texts, probs):
    print(f"{p.item():.3f}  {t}")

> **관찰 포인트 ★**: 이미지 벡터와 텍스트 벡터가 **똑같이 512차원**입니다. **그래서 내적이 가능합니다.** 이것이 §0에서 말한 *"같은 자로 잰다"* 의 실체입니다.

**셀 3** — 유사도를 직접 계산해 본다 (라이브러리 없이)

In [ ]:
import torch.nn.functional as F
i_emb = F.normalize(out.image_embeds, dim=-1)        # (1, 512)
t_emb = F.normalize(out.text_embeds,  dim=-1)        # (4, 512)
cos   = (i_emb @ t_emb.T)[0]                         # ★ 코사인 유사도 (4,)

for t, c in zip(texts, cos):
    print(f"{c.item():+.4f}  {t}")
print("\nlogits_per_image = 코사인 × logit_scale :", round(model.logit_scale.exp().item(), 2))

> **핵심**: **11주차 어텐션의 `QKᵀ` 와 같은 계산**입니다 — 내적으로 유사도를 재고, softmax 로 비중을 만듭니다. **또 나왔습니다.** `logit_scale` 은 softmax 를 날카롭게 만드는 학습된 온도값입니다.

## 2-2. 구현

**셀 4** — FashionMNIST 로 먼저 감을 잡는다

In [ ]:
from torchvision import datasets, transforms
CLASSES = ["t-shirt", "trousers", "pullover", "dress", "coat",
           "sandal", "shirt", "sneaker", "bag", "ankle boot"]
prompts = [f"a photo of a {c}" for c in CLASSES]        # ★ 프롬프트 템플릿

te = datasets.FashionMNIST("data", train=False, transform=transforms.ToTensor())

def zeroshot(pil_images, prompts):
    inp = proc(text=prompts, images=pil_images, return_tensors="pt", padding=True).to(DEV)
    with torch.no_grad():
        return model(**inp).logits_per_image.softmax(-1)     # (N_img, N_class)

imgs = [transforms.ToPILImage()(te[i][0]).convert("RGB") for i in range(200)]
ys   = np.array([te[i][1] for i in range(200)])
p    = zeroshot(imgs, prompts).cpu().numpy()
pred = p.argmax(-1)
print(f"제로샷 정확도 (FashionMNIST 200장) : {(pred == ys).mean():.3f}")

> **관찰 포인트 ★**: 정확도가 **60~70% 정도**로 낮게 나옵니다. FashionMNIST 는 **28×28 흑백**이라 CLIP 이 학습한 인터넷 사진과 성격이 매우 다릅니다. *"제로샷은 만능이 아니다"* 를 보여 주는 좋은 사례입니다.

**셀 5** — 프롬프트를 바꾸면 정확도가 변한다 ★

In [ ]:
templates = ["{}",
             "a photo of a {}",
             "a photo of a {}, a type of clothing",
             "a low resolution grayscale photo of a {}"]
for tp in templates:
    pr = [tp.format(c) for c in CLASSES]
    acc = (zeroshot(imgs, pr).cpu().numpy().argmax(-1) == ys).mean()
    print(f"{acc:.3f}   \"{tp}\"")

> **핵심 ★★**: **같은 모델·같은 이미지인데 프롬프트만 바꿔도 정확도가 달라집니다.** 이것이 **프롬프트 엔지니어링**이고, 제로샷의 **강점이자 불안정성**입니다. *"학습 대신 프롬프트를 튜닝하는 것"* — 3학년 「최신인공지능」에서 본격적으로 다룹니다.

**셀 6** — 내 데이터셋(9주차)으로 ★★

In [ ]:
import glob, os
MYDATA = "mydata/val"                                  # 9주차 커스텀 데이터셋
my_classes = sorted(os.listdir(MYDATA))                # 폴더명 = 클래스명
print("내 클래스 :", my_classes)

my_prompts = [f"a photo of a {c}" for c in my_classes]  # ★ 폴더명을 그대로 문장으로
files, labels = [], []
for ci, c in enumerate(my_classes):
    fs = glob.glob(os.path.join(MYDATA, c, "*"))
    files += fs; labels += [ci] * len(fs)
labels = np.array(labels)

my_imgs = [Image.open(f).convert("RGB") for f in files]
zs_pred = zeroshot(my_imgs, my_prompts).cpu().numpy().argmax(-1)
zs_acc  = (zs_pred == labels).mean()
print(f"\n제로샷 정확도 (내 데이터 {len(files)}장) : {zs_acc:.3f}")
print("학습 데이터 0장, 학습 시간 0초 ★")

## 3. 실습 8 — 9주차 파인튜닝 결과와 비교 ★★

**셀 7** — 세 방식을 한 표로 ★★

In [ ]:
rows = [
    # 방식,               정확도,        준비 데이터, 학습 시간, GPU 메모리
    ("9주차 백본 동결",   0.00,          "200장",     "3분",     "2GB"),
    ("9주차 LoRA",        0.00,          "200장",     "5분",     "3GB"),
    ("14주차 제로샷",     round(zs_acc,3), "0장",     "0초",     "1GB"),
]
print(f"{'방식':18s}{'정확도':>8s}{'데이터':>10s}{'학습시간':>10s}{'메모리':>8s}")
for r in rows:
    print(f"{r[0]:18s}{r[1]:>8}{r[2]:>10s}{r[3]:>10s}{r[4]:>8s}")
print("\n※ 9주차 값은 본인 결과로 채워 넣을 것 ★")

**셀 8** — 결과를 파일로 (과제 제출물 ★)

In [ ]:
report = f"""# 제로샷 vs 파인튜닝 비교
- 데이터셋: 내 커스텀 {len(my_classes)}클래스 {len(files)}장
- 제로샷(CLIP ViT-B/32) 정확도: {zs_acc:.3f}  (학습 데이터 0장, 학습 0초)
- 9주차 백본 동결 정확도: (기입)
- 9주차 LoRA 정확도: (기입)

## 5줄 판단
1.
2.
3.
4.
5.
"""
os.makedirs("results", exist_ok=True)
open("results/zeroshot_vs_finetune.md", "w", encoding="utf-8").write(report)
print(report)